# Workshop 2: Retrieval Basics with RAG
## Theme Example: Health Information (MedlinePlus XML)

In this notebook, we work with a health dataset in XML format.

XML is a structured format that stores information using tags.  
We will extract useful text from the XML file, prepare it for retrieval, and build a simple vector search system.

Our goal is to:
- load the XML data  
- extract useful text fields  
- split the text into chunks  
- create embeddings  
- store them in a vector database  
- retrieve relevant information for a query

## Working with XML Data

Unlike plain text or PDFs, XML data is organised using tags.

This means we first need to extract the parts we want to use.

For this workshop, we will focus on key text fields such as:
- title  
- summary  
- full article text (if available)

In [1]:
# Install required libraries (run once)
!pip install -q --upgrade langchain langchain-community langchain-text-splitters sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.


In [2]:
import requests
import xml.etree.ElementTree as ET

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [3]:
url = "https://medlineplus.gov/xml/mplus_topics_2026-03-21.xml"

response = requests.get(url)

with open("medlineplus.xml", "wb") as f:
    f.write(response.content)

print("XML file downloaded successfully!")

XML file downloaded successfully!


In [4]:
from lxml import etree

parser = etree.XMLParser(recover=True)
tree = etree.parse("/content/medlineplus.xml", parser)
root = tree.getroot()

print("Root tag:", root.tag)

Root tag: health-topics


## Extracting Text from XML

The XML file contains many fields, but we only need the main text content.

For each topic, we extract:
- title  
- summary  

We combine them into a single text that will be used for retrieval.

In [5]:
from bs4 import BeautifulSoup

documents = []

for topic in root.findall(".//health-topic"):
    title = topic.findtext("title", default="")
    summary = topic.findtext("full-summary", default="")

    clean_summary = BeautifulSoup(summary, "html.parser").get_text(" ", strip=True)
    text = title + " " + clean_summary
    documents.append(text)

print("Number of documents:", len(documents))
print(documents[0][:500])

Number of documents: 2033
 A1C is a blood test for type 2 diabetes and prediabetes . It measures your average blood glucose, or blood sugar , level over the past 3 months. Doctors may use the A1C alone or in combination with other diabetes tests to make a diagnosis. They also use the A1C to see how well you are managing your diabetes. This test is different from the blood sugar checks that people with diabetes do every day. Your A1C test result is given in percentages. The higher the percentage, the higher your blood sug


## Splitting the Health Topics into Chunks

The extracted health topics can still be long, so we split them into smaller chunks.

This helps the retrieval system search more effectively and return more focused results.

In [6]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.create_documents(documents)

print("Number of chunks:", len(chunks))
print(chunks[0].page_content[:500])

Number of chunks: 5695
A1C is a blood test for type 2 diabetes and prediabetes . It measures your average blood glucose, or blood sugar , level over the past 3 months. Doctors may use the A1C alone or in combination with other diabetes tests to make a diagnosis. They also use the A1C to see how well you are managing your diabetes. This test is different from the blood sugar checks that people with diabetes do every day. Your A1C test result is given in percentages. The higher the percentage, the higher your blood suga


## Creating Embeddings and Building the Vector Store

Now we convert each chunk into an embedding, which is a numerical representation of its meaning.

We then store these embeddings in FAISS, so that we can quickly retrieve the most relevant chunks for a query.

In [7]:
embedding_model = HuggingFaceEmbeddings()

db = FAISS.from_documents(chunks, embedding_model)

print("Vector store created successfully!")

/tmp/ipykernel_4680/3680806366.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings()
/tmp/ipykernel_4680/3680806366.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding_model = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/s

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store created successfully!


## Retrieving Relevant Information

We can now search the health topics using a query.

The system compares the query with the stored embeddings and returns the most relevant chunks.

In [8]:
query = "symptoms of diabetes"

results = db.similarity_search(query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n")
    print(r.page_content[:500])


Result 1:

Diabetes means your blood glucose, or blood sugar , levels are too 
high.  With type 1 diabetes, your pancreas does not make insulin. Insulin is a hormone that 
helps glucose get into your cells to give them energy. Without insulin, too much glucose 
stays in your blood.  Over time, high blood glucose can lead to serious problems with your heart , eyes , kidneys , nerves , and gums and teeth. Type 1 diabetes happens most often in children and young adults but can appear at any 
age. Symptoms may

Result 2:

A key part of managing diabetes is controlling your blood glucose levels. To do this, you need to follow a diabetes meal plan and get regular physical activity . You might also need to take diabetes medicines . You have to balance all of these to keep your blood glucose at the right levels. But if you eat too much food or the wrong foods, don't take your medicines correctly, or don't get physical activity, you can get hyperglycemia. It can also happen if you are stressed

In [9]:
# Save FAISS index locally
db.save_local("faiss_Health")

## Trying Different Queries

Try a few different health-related queries and observe the retrieved results.

Think about:
- Are the results relevant?
- Do different queries return different types of information?
- Which query seems more precise?

## Summary

In this notebook, we:
- loaded a health dataset in XML format
- extracted useful text fields
- cleaned HTML content
- split the text into chunks
- created embeddings
- stored them in a vector database
- retrieved relevant information using queries

This shows how retrieval can be applied to structured health information.